# End-to-end Pipeline

End-to-end notebook for *"When More Becomes Less: A Two-Probe
Diagnostic for Repetition in Language Models."* Runs both panels in
one go:

1. **English panel** — 13 open-access checkpoints (MLM + CLM, 0.5B–14B).
   Two-probe displaced grid F0–F4; adjacent baseline on 3 models;
   six-condition ablation + mechanism + layerwise + LAMA-style pilot
   on the 4 mechanism models; aggregation into `summary/`.
2. **Multilingual panel** — 4 multilingual checkpoints (XLM-R-base,
   XLM-R-large, Qwen2.5-1.5B, Qwen2.5-7B) × 4 languages (Spanish,
   Chinese, German, French) with F0–F2 hand-translated frames. XLM-R
   Chinese drops out because no Chinese single-token candidate
   survives the BPE filter, leaving 14 (model, language) combinations
   and 42 evaluated cells.

The notebook is restartable: each (model, panel) cell skips work
whose output is already on disk.

## §0 Setup

In [ ]:
# Dependency install. We pip-install everything (including numpy upgrade)
# and then restart the kernel so the freshly-upgraded packages are loaded
# cleanly. After the restart, re-run all cells from the top — this cell
# becomes a no-op on the second pass (sentinel file detected).
import os, sys, subprocess
from pathlib import Path
SENTINEL = Path("/tmp/_lexrep_deps_installed")
if SENTINEL.exists():
    print("dependencies already installed (sentinel found); continuing.")
else:
    DEPS = [
        "numpy>=2.1",
        "transformers>=4.48",
        "accelerate",
        "bitsandbytes",
        "pandas",
        "scipy",
        "statsmodels",
        "pyarrow",
        "wordfreq",
    ]
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + DEPS
    )
    SENTINEL.touch()
    print("dependencies installed.")
    print("RESTARTING KERNEL so the new versions load cleanly.")
    print("After the restart, re-run all cells from the top.")
    os.kill(os.getpid(), 9)


In [ ]:
import os, sys, json, gc, time, math, random, traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM

# --- Output roots ---
# In Colab, mount Drive and write into a stable folder. Locally, the
# notebook runs from `notebooks/` so default to ../results/.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    RESULTS = Path('/content/drive/MyDrive/repeated_lexical_results')
except Exception as e:
    print(f"[info] not in Colab ({e!s}); writing to ../results/")
    RESULTS = Path(__file__).resolve().parent.parent / 'results' \
        if '__file__' in globals() else Path('../results').resolve()

DRIVE_BASE = RESULTS / 'english_panel'        # English panel root
ML_DIR     = RESULTS / 'multilingual_panel'   # Multilingual panel root

DRIVE_BASE.mkdir(parents=True, exist_ok=True)
ML_DIR.mkdir(parents=True, exist_ok=True)
for sub in ['targets','displaced','adjacent','ablation','mechanism',
            'application','layerwise','summary']:
    (DRIVE_BASE/sub).mkdir(exist_ok=True)
(DRIVE_BASE/'summary'/'logs').mkdir(exist_ok=True)
for sub in ['targets','displaced','summary']:
    (ML_DIR/sub).mkdir(exist_ok=True)
(ML_DIR/'summary'/'logs').mkdir(exist_ok=True)

print(f"English panel  → {DRIVE_BASE}")
print(f"Multilingual   → {ML_DIR}")
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'NONE'}")
if torch.cuda.is_available():
    print(f"GPU mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
import transformers as _t
print(f"transformers: {_t.__version__}")
torch.set_grad_enabled(False)


## §1 English panel

13-model open-access run: wordlist + targets + displaced + adjacent
+ ablation + mechanism + layerwise + LAMA-style application probe +
aggregation.

In [ ]:
# Embedded data: 261-word target list + semantic-neighbour dict + filler pools.
import base64, io

WORDLIST_B64 = """d29yZCxjYXRlZ29yeSxjb25jcmV0ZW5lc3NfZ3JvdXAsZnJlcXVlbmN5X2dyb3VwLHNlbWFudGljX2NsYXNzLHdvcmRfbGVuZ3RoLHppcGZfZnJlcXVlbmN5LG5vdGVzCmJyZWFkLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLGZvb2QsNSw0LjUsCndhdGVyLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsdmVyeV9oaWdoLGZvb2QsNSw1LjUyLAptaWxrLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLGZvb2QsNCw0LjY2LApyaWNlLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLGZvb2QsNCw0LjU1LAptZWF0LGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLGZvb2QsNCw0LjYzLAplZ2csY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsZm9vZCwzLDQuNDYsCnNhbHQsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsZm9vZCw0LDQuNiwKc3VnYXIsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsZm9vZCw1LDQuNjgsCmNoZWVzZSxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCxmb29kLDYsNC41NywKdGVhLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLGZvb2QsMyw0LjczLApjb2ZmZWUsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLGZvb2QsNiw0Ljg2LApzb3VwLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLGZvb2QsNCw0LjE1LAphcHBsZSxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCxmb29kLDUsNC43NiwKZnJ1aXQsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsZm9vZCw1LDQuNTcsCmZvb2QsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLGZvb2QsNCw1LjQsCmRvZyxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsYW5pbWFsLDMsNS4xLApjYXQsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsYW5pbWFsLDMsNC43OCwKaG9yc2UsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsYW5pbWFsLDUsNC43NiwKYmlyZCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCxhbmltYWwsNCw0LjYzLApmaXNoLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxhbmltYWwsNCw0LjksCnNoZWVwLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLGFuaW1hbCw1LDQuMTksCnBpZyxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCxhbmltYWwsMyw0LjE0LApjb3csY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsYW5pbWFsLDMsNC4xNCwKaG91c2UsY29uY3JldGVfaGlnaCxjb25jcmV0ZSx2ZXJ5X2hpZ2gsb2JqZWN0LDUsNS43MSwKY2FyLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxvYmplY3QsMyw1LjQ1LApkb29yLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxvYmplY3QsNCw1LjA4LAp0YWJsZSxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsb2JqZWN0LDUsNS4wNSwKY2hhaXIsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsb2JqZWN0LDUsNC42OSwKYmVkLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxvYmplY3QsMyw1LjA3LApib29rLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxvYmplY3QsNCw1LjQzLApwYXBlcixjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsb2JqZWN0LDUsNS4wNywKcGhvbmUsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLG9iamVjdCw1LDUuMywKd2luZG93LGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxvYmplY3QsNiw0LjgsCmJveCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsb2JqZWN0LDMsNS4wNCwKYmFnLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLG9iamVjdCwzLDQuNzksCmN1cCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsb2JqZWN0LDMsNS4xMSwKa25pZmUsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxtaWQsdG9vbCw1LDQuNDEsCmhhbW1lcixjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCx0b29sLDYsNC4xNiwKa2V5LGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxvYmplY3QsMyw1LjEyLApyb29tLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxwbGFjZSw0LDUuMzksCmNpdHksY29uY3JldGVfaGlnaCxjb25jcmV0ZSx2ZXJ5X2hpZ2gscGxhY2UsNCw1LjYxLApzY2hvb2wsY29uY3JldGVfaGlnaCxjb25jcmV0ZSx2ZXJ5X2hpZ2gscGxhY2UsNiw1LjcxLApwYXJrLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxwbGFjZSw0LDUuMTYsCmdhcmRlbixjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCxwbGFjZSw2LDQuNzcsCm9mZmljZSxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gscGxhY2UsNiw1LjQsCnN0cmVldCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gscGxhY2UsNiw1LjI4LApyb2FkLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxwbGFjZSw0LDUuMjQsCmhhbmQsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLGJvZHksNCw1LjQxLApleWUsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLGJvZHksMyw1LjAyLApmYWNlLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxib2R5LDQsNS40NSwKZm9vdCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsYm9keSw0LDQuODcsCmFybSxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCxib2R5LDMsNC43MywKbGVnLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLGJvZHksMyw0LjY2LApoZWFkLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsdmVyeV9oaWdoLGJvZHksNCw1LjUxLApoZWFydCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsYm9keSw1LDUuMzEsCmhhaXIsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLGJvZHksNCw1LjA4LAptb3V0aCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsYm9keSw1LDQuODMsCnRyZWUsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLG5hdHVyYWxfb2JqZWN0LDQsNC44NSwKc3RvbmUsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLG5hdHVyYWxfb2JqZWN0LDUsNC44LApyb2NrLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxuYXR1cmFsX29iamVjdCw0LDUuMDMsCnJpdmVyLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxuYXR1cmFsX29iamVjdCw1LDUuMDMsCnNreSxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCxuYXR1cmFsX29iamVjdCwzLDQuNzIsCnN1bixjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsbmF0dXJhbF9vYmplY3QsMyw0Ljk3LAptb29uLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLG5hdHVyYWxfb2JqZWN0LDQsNC43LApjbG91ZCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCxuYXR1cmFsX29iamVjdCw1LDQuNDYsCmZpcmUsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLG5hdHVyYWxfb2JqZWN0LDQsNS4zLAp3b29kLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLG5hdHVyYWxfb2JqZWN0LDQsNC43OCwKZWFydGgsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLG5hdHVyYWxfb2JqZWN0LDUsNS4wNiwKcGVuLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLHRvb2wsMyw0LjM4LApjbG9jayxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCx0b29sLDUsNC40MywKd2F0Y2gsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLHRvb2wsNSw1LjM0LApjYW1lcmEsY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLHRvb2wsNiw0Ljg3LApicnVzaCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLG1pZCx0b29sLDUsNC4yMywKbGFkZGVyLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsbWlkLHRvb2wsNiw0LjAzLAptYW4sY29uY3JldGVfaGlnaCxjb25jcmV0ZSx2ZXJ5X2hpZ2gsb2JqZWN0LDMsNS44MiwKd29tYW4sY29uY3JldGVfaGlnaCxjb25jcmV0ZSxoaWdoLG9iamVjdCw1LDUuMzUsCmNoaWxkLGNvbmNyZXRlX2hpZ2gsY29uY3JldGUsaGlnaCxvYmplY3QsNSw1LjMsCmJveSxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsb2JqZWN0LDMsNS4xNywKZ2lybCxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsb2JqZWN0LDQsNS4zOCwKYmFieSxjb25jcmV0ZV9oaWdoLGNvbmNyZXRlLGhpZ2gsb2JqZWN0LDQsNS4yNiwKdmlvbGluLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csdG9vbCw2LDMuNzQsCmxhbnRlcm4sY29uY3JldGVfbG93LGNvbmNyZXRlLGxvdyxvYmplY3QsNywzLjYxLApjYWN0dXMsY29uY3JldGVfbG93LGNvbmNyZXRlLGxvdyxuYXR1cmFsX29iamVjdCw2LDMuNCwKaGVsbWV0LGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsb2JqZWN0LDYsNC4wMiwKcGVhcmwsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxvYmplY3QsNSw0LjE0LAphbmNob3IsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxvYmplY3QsNiw0LjAyLApzaG92ZWwsY29uY3JldGVfbG93LGNvbmNyZXRlLGxvdyx0b29sLDYsMy41LAprZXR0bGUsY29uY3JldGVfbG93LGNvbmNyZXRlLGxvdyx0b29sLDYsMy41NCwKY29tcGFzcyxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LHRvb2wsNywzLjc0LAp0ZWxlc2NvcGUsY29uY3JldGVfbG93LGNvbmNyZXRlLGxvdyx0b29sLDksMy43NSwKbWljcm9zY29wZSxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LHRvb2wsMTAsMy41LApoYW1tb2NrLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csb2JqZWN0LDcsMy4xNCwKcmliYm9uLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csb2JqZWN0LDYsMy44MSwKYnV0dG9uLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsb2JqZWN0LDYsNC41NiwKY3JheW9uLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csdG9vbCw2LDMuMDUsCnBpYW5vLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsdG9vbCw1LDQuMzEsCmVhZ2xlLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsYW5pbWFsLDUsNC4xMywKdGlnZXIsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxhbmltYWwsNSw0LjMsCmNhbWVsLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csYW5pbWFsLDUsMy42NCwKc25ha2UsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxhbmltYWwsNSw0LjIzLApyYWJiaXQsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxhbmltYWwsNiw0LjA1LAp0dXJ0bGUsY29uY3JldGVfbG93LGNvbmNyZXRlLGxvdyxhbmltYWwsNiwzLjkxLApmb3gsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxhbmltYWwsMyw0LjY2LApkZWVyLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsYW5pbWFsLDQsNC4xMiwKZnJvZyxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LGFuaW1hbCw0LDMuODksCmJlZSxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLGFuaW1hbCwzLDQuMDksCmFudCxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LGFuaW1hbCwzLDMuODYsCm93bCxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LGFuaW1hbCwzLDMuODcsCmNyb3csY29uY3JldGVfbG93LGNvbmNyZXRlLGxvdyxhbmltYWwsNCwzLjgsCndoYWxlLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csYW5pbWFsLDUsMy45OCwKc3Bvb24sY29uY3JldGVfbG93LGNvbmNyZXRlLGxvdyx0b29sLDUsMy44OCwKZm9yayxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLHRvb2wsNCw0LjAzLApwbGF0ZSxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLG9iamVjdCw1LDQuNTUsCmJvd2wsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxvYmplY3QsNCw0LjYzLAp2YXNlLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csb2JqZWN0LDQsMy41NCwKbWlycm9yLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsb2JqZWN0LDYsNC40NywKcm9wZSxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLHRvb2wsNCw0LjA2LApibGFua2V0LGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsb2JqZWN0LDcsNC4wNywKcGlsbG93LGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csb2JqZWN0LDYsMy45NiwKY2FycGV0LGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsb2JqZWN0LDYsNC4xOCwKZHJ1bSxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLHRvb2wsNCw0LjExLAphbmtsZSxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LGJvZHksNSwzLjk3LAplbGJvdyxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LGJvZHksNSwzLjg0LAprbmVlLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsYm9keSw0LDQuMzMsCndyaXN0LGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsYm9keSw1LDQuMDEsCmxpdmVyLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQsYm9keSw1LDQuMTgsCmtpZG5leSxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLGJvZHksNiw0LjAzLApzcGluZSxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LGJvZHksNSwzLjkzLAp0ZW1wbGUsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxwbGFjZSw2LDQuNDUsCmNhc3RsZSxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLHBsYWNlLDYsNC40NSwKY290dGFnZSxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLHBsYWNlLDcsNC4wLApkZXNlcnQsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxwbGFjZSw2LDQuNCwKZm9yZXN0LGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxtaWQscGxhY2UsNiw0LjcsCmhhcmJvcixjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLHBsYWNlLDYsNC4xNSwKdmlsbGFnZSxjb25jcmV0ZV9sb3csY29uY3JldGUsaGlnaCxwbGFjZSw3LDQuODIsCnZhbGxleSxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLHBsYWNlLDYsNC43MiwKY2FueW9uLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3cscGxhY2UsNiwzLjk2LApjb21ldCxjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LG5hdHVyYWxfb2JqZWN0LDUsMy42NCwKY3J5c3RhbCxjb25jcmV0ZV9sb3csY29uY3JldGUsbWlkLG5hdHVyYWxfb2JqZWN0LDcsNC40LApnbGFjaWVyLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csbmF0dXJhbF9vYmplY3QsNywzLjU4LApwbGFuZXQsY29uY3JldGVfbG93LGNvbmNyZXRlLG1pZCxuYXR1cmFsX29iamVjdCw2LDQuNjgsCmNyYXRlcixjb25jcmV0ZV9sb3csY29uY3JldGUsbG93LG5hdHVyYWxfb2JqZWN0LDYsMy41NSwKcGViYmxlLGNvbmNyZXRlX2xvdyxjb25jcmV0ZSxsb3csbmF0dXJhbF9vYmplY3QsNiwzLjM0LApsb3ZlLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsdmVyeV9oaWdoLGVtb3Rpb24sNCw1LjgyLApmZWFyLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxlbW90aW9uLDQsNC45NSwKaG9wZSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsZW1vdGlvbiw0LDUuNDQsCmpveSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LG1pZCxlbW90aW9uLDMsNC41NSwKYW5nZXIsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxtaWQsZW1vdGlvbiw1LDQuNDIsCmhhdGUsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGVtb3Rpb24sNCw1LjExLApwYWluLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxlbW90aW9uLDQsNS4wMywKcHJpZGUsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxtaWQsZW1vdGlvbiw1LDQuNDksCnRydXN0LGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxlbW90aW9uLDUsNS4xMywKaWRlYSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDQsNS4zNiwKbWluZCxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDQsNS40MSwKdGhvdWdodCxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LHZlcnlfaGlnaCxjb2duaXRpb24sNyw1LjU5LAptZW1vcnksYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGNvZ25pdGlvbiw2LDQuODEsCmRyZWFtLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxjb2duaXRpb24sNSw0LjkxLAprbm93bGVkZ2UsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGNvZ25pdGlvbiw5LDQuOTYsCnJlYXNvbixhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDYsNS4zMiwKYmVsaWVmLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsbWlkLGNvZ25pdGlvbiw2LDQuNDYsCnF1ZXN0aW9uLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxjb2duaXRpb24sOCw1LjM1LApmYW1pbHksYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCx2ZXJ5X2hpZ2gsc29jaWFsX2NvbmNlcHQsNiw1LjY2LApmcmllbmQsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLHNvY2lhbF9jb25jZXB0LDYsNS4zNywKaG9tZSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LHZlcnlfaGlnaCxzb2NpYWxfY29uY2VwdCw0LDUuODEsCmxpZmUsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCx2ZXJ5X2hpZ2gsc29jaWFsX2NvbmNlcHQsNCw1Ljg5LApkZWF0aCxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsc29jaWFsX2NvbmNlcHQsNSw1LjQsCndvcmssYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCx2ZXJ5X2hpZ2gsc29jaWFsX2NvbmNlcHQsNCw1Ljk2LAptb25leSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LHZlcnlfaGlnaCxzb2NpYWxfY29uY2VwdCw1LDUuNjQsCnRpbWUsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCx2ZXJ5X2hpZ2gsc29jaWFsX2NvbmNlcHQsNCw2LjI5LApoaXN0b3J5LGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxzb2NpYWxfY29uY2VwdCw3LDUuMzksCmN1bHR1cmUsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLHNvY2lhbF9jb25jZXB0LDcsNS4wMywKbGFuZ3VhZ2UsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLHNvY2lhbF9jb25jZXB0LDgsNS4xLAptdXNpYyxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LHZlcnlfaGlnaCxzb2NpYWxfY29uY2VwdCw1LDUuNTIsCmFydCxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsc29jaWFsX2NvbmNlcHQsMyw1LjI5LApwb3dlcixhYnN0cmFjdF9oaWdoLGFic3RyYWN0LHZlcnlfaGlnaCxwb2xpdGljYWxfY29uY2VwdCw1LDUuNTIsCmZyZWVkb20sYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLHBvbGl0aWNhbF9jb25jZXB0LDcsNC44NiwKbGF3LGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxwb2xpdGljYWxfY29uY2VwdCwzLDUuNDYsCnJpZ2h0LGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsdmVyeV9oaWdoLHBvbGl0aWNhbF9jb25jZXB0LDUsNS45NiwKcGVhY2UsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLHBvbGl0aWNhbF9jb25jZXB0LDUsNS4wMiwKd2FyLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxwb2xpdGljYWxfY29uY2VwdCwzLDUuNDYsCmdvdmVybm1lbnQsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCx2ZXJ5X2hpZ2gscG9saXRpY2FsX2NvbmNlcHQsMTAsNS41NywKdHJ1dGgsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLG1vcmFsX2NvbmNlcHQsNSw1LjA1LApnb29kLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsdmVyeV9oaWdoLG1vcmFsX2NvbmNlcHQsNCw2LjEyLApldmlsLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsbWlkLG1vcmFsX2NvbmNlcHQsNCw0Ljc0LApob25vcixhYnN0cmFjdF9oaWdoLGFic3RyYWN0LG1pZCxtb3JhbF9jb25jZXB0LDUsNC42OSwKanVzdGljZSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsbW9yYWxfY29uY2VwdCw3LDQuOTcsCmNoYW5nZSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LHZlcnlfaGlnaCxjb2duaXRpb24sNiw1LjU0LApmdXR1cmUsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGNvZ25pdGlvbiw2LDUuMzMsCnJlYWxpdHksYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGNvZ25pdGlvbiw3LDQuODksCm5hdHVyZSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDYsNC45OCwKc3lzdGVtLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsdmVyeV9oaWdoLGNvZ25pdGlvbiw2LDUuNTYsCnByb2JsZW0sYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGNvZ25pdGlvbiw3LDUuNCwKYW5zd2VyLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxjb2duaXRpb24sNiw1LjE3LApzdG9yeSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDUsNS40NSwKbmFtZSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LHZlcnlfaGlnaCxjb2duaXRpb24sNCw1LjYxLApudW1iZXIsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCx2ZXJ5X2hpZ2gsY29nbml0aW9uLDYsNS42MiwKd29yZCxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDQsNS4yNiwKdm9pY2UsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGNvZ25pdGlvbiw1LDUuMDcsCnNvdW5kLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxjb2duaXRpb24sNSw1LjE1LApsaWdodCxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDUsNS4zMywKZGFyayxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDQsNS4wNCwKY29sb3IsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGNvZ25pdGlvbiw1LDQuOTEsCnNoYXBlLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsbWlkLGNvZ25pdGlvbiw1LDQuNzIsCnNpemUsYWJzdHJhY3RfaGlnaCxhYnN0cmFjdCxoaWdoLGNvZ25pdGlvbiw0LDUuMTMsCnNwYWNlLGFic3RyYWN0X2hpZ2gsYWJzdHJhY3QsaGlnaCxjb2duaXRpb24sNSw1LjIzLApwbGFjZSxhYnN0cmFjdF9oaWdoLGFic3RyYWN0LHZlcnlfaGlnaCxzb2NpYWxfY29uY2VwdCw1LDUuNzEsCmFueGlldHksYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxlbW90aW9uLDcsNC40NSwKZW52eSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LGVtb3Rpb24sNCwzLjg0LApncmllZixhYnN0cmFjdF9sb3csYWJzdHJhY3QsbWlkLGVtb3Rpb24sNSw0LjA4LApyYWdlLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsZW1vdGlvbiw0LDQuMTEsCnNvcnJvdyxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LGVtb3Rpb24sNiwzLjczLApwaXR5LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxsb3csZW1vdGlvbiw0LDMuOTgsCnJlZ3JldCxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbWlkLGVtb3Rpb24sNiw0LjM3LApkaXNndXN0LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxsb3csZW1vdGlvbiw3LDMuNjEsCmRlbGlnaHQsYWJzdHJhY3RfbG93LGFic3RyYWN0LGxvdyxlbW90aW9uLDcsMy45NywKbG9uZ2luZyxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LGVtb3Rpb24sNywzLjUsCmNvbnRlbXB0LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxsb3csZW1vdGlvbiw4LDMuNzQsCndpc2RvbSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbWlkLGNvZ25pdGlvbiw2LDQuMywKaW50dWl0aW9uLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxsb3csY29nbml0aW9uLDksMy41NCwKcGFyYWRveCxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LGNvZ25pdGlvbiw3LDMuNjMsCmxvZ2ljLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsY29nbml0aW9uLDUsNC4zNiwKaW50ZW50LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsY29nbml0aW9uLDYsNC4yOCwKZG91YnQsYWJzdHJhY3RfbG93LGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDUsNC45LApjbGFyaXR5LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxsb3csY29nbml0aW9uLDcsMy45LApsb3lhbHR5LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsc29jaWFsX2NvbmNlcHQsNyw0LjE3LApraW5zaGlwLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxsb3csc29jaWFsX2NvbmNlcHQsNywzLjEzLApzb2xpdHVkZSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LHNvY2lhbF9jb25jZXB0LDgsMy40MiwKcml0dWFsLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsc29jaWFsX2NvbmNlcHQsNiw0LjAyLAp0cmFkaXRpb24sYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxzb2NpYWxfY29uY2VwdCw5LDQuNSwKbGVnZW5kLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsc29jaWFsX2NvbmNlcHQsNiw0LjQ0LApjdXN0b20sYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxzb2NpYWxfY29uY2VwdCw2LDQuNDMsCmRlbW9jcmFjeSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbWlkLHBvbGl0aWNhbF9jb25jZXB0LDksNC41MSwKcmVwdWJsaWMsYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxwb2xpdGljYWxfY29uY2VwdCw4LDQuNiwKdHJlYXR5LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQscG9saXRpY2FsX2NvbmNlcHQsNiw0LjMxLApkb2N0cmluZSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LHBvbGl0aWNhbF9jb25jZXB0LDgsMy45OCwKcmVnaW1lLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQscG9saXRpY2FsX2NvbmNlcHQsNiw0LjMsCnBvbGljeSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsaGlnaCxwb2xpdGljYWxfY29uY2VwdCw2LDUuMTcsCmF1dGhvcml0eSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsaGlnaCxwb2xpdGljYWxfY29uY2VwdCw5LDQuODEsCmxpYmVydHksYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxwb2xpdGljYWxfY29uY2VwdCw3LDQuMzQsCnR5cmFubnksYWJzdHJhY3RfbG93LGFic3RyYWN0LGxvdyxwb2xpdGljYWxfY29uY2VwdCw3LDMuNTIsCm1lcmN5LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsbW9yYWxfY29uY2VwdCw1LDQuMjUsCnZpcnR1ZSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LG1vcmFsX2NvbmNlcHQsNiwzLjk0LAp2aWNlLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsbW9yYWxfY29uY2VwdCw0LDQuNzEsCmR1dHksYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxtb3JhbF9jb25jZXB0LDQsNC43MSwKc2hhbWUsYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxtb3JhbF9jb25jZXB0LDUsNC41NSwKZ3VpbHQsYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxtb3JhbF9jb25jZXB0LDUsNC4xMywKZGlnbml0eSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbWlkLG1vcmFsX2NvbmNlcHQsNyw0LjAyLAppbnRlZ3JpdHksYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxtb3JhbF9jb25jZXB0LDksNC4yMiwKYW1iaXRpb24sYWJzdHJhY3RfbG93LGFic3RyYWN0LGxvdyxjb2duaXRpb24sOCwzLjkxLApkZXN0aW55LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsY29nbml0aW9uLDcsNC4xOSwKZXNzZW5jZSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbWlkLGNvZ25pdGlvbiw3LDQuMDMsCm9yaWdpbixhYnN0cmFjdF9sb3csYWJzdHJhY3QsbWlkLGNvZ25pdGlvbiw2LDQuNTUsCnB1cnBvc2UsYWJzdHJhY3RfbG93LGFic3RyYWN0LGhpZ2gsY29nbml0aW9uLDcsNC44OCwKbm90aW9uLGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsY29nbml0aW9uLDYsNC4xNywKbGVnYWN5LGFic3RyYWN0X2xvdyxhYnN0cmFjdCxtaWQsc29jaWFsX2NvbmNlcHQsNiw0LjM3LApidXJkZW4sYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxjb2duaXRpb24sNiw0LjIzLApzeW1ib2wsYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxjb2duaXRpb24sNiw0LjMxLAptZXRhcGhvcixhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LGNvZ25pdGlvbiw4LDMuNzMsCnBlcmNlcHRpb24sYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxjb2duaXRpb24sMTAsNC4xOCwKaW1hZ2luYXRpb24sYWJzdHJhY3RfbG93LGFic3RyYWN0LG1pZCxjb2duaXRpb24sMTEsNC4yMiwKY29uc2NpZW5jZSxhYnN0cmFjdF9sb3csYWJzdHJhY3QsbG93LGNvZ25pdGlvbiwxMCwzLjk2LAo="""

WORDLIST_CSV = base64.b64decode(WORDLIST_B64).decode()
WORDLIST_DF = pd.read_csv(io.StringIO(WORDLIST_CSV))
print(f"loaded {len(WORDLIST_DF)} candidate words")
print(WORDLIST_DF.head(3))

SEM_FIELD = {
    "time": ["day","night","hour","minute","year"],
    "day": ["night","week","time","morning"],
    "man": ["woman","boy","person","human"],
    "house": ["home","building","apartment","room"],
    "water": ["liquid","river","ocean","sea","drink"],
    "food": ["meal","dinner","lunch","bread","drink"],
    "violin": ["music","instrument","cello","guitar"],
    "lantern": ["light","lamp","candle","fire"],
    "cactus": ["plant","desert","tree","flower"],
    "helmet": ["hat","head","armor","protection"],
    "pearl": ["jewel","gem","white","necklace"],
    "idea": ["thought","concept","plan","mind"],
    "love": ["hate","emotion","heart","romance"],
    "power": ["force","energy","strength","control"],
    "truth": ["lie","fact","reality","honesty"],
    "freedom": ["liberty","independence","right","choice"],
    "anger": ["fear","hate","emotion","rage"],
    "justice": ["law","fairness","truth","equity"],
    "democracy": ["freedom","government","politics","liberty"],
    "wisdom": ["knowledge","intelligence","experience","insight"],
    "anxiety": ["fear","worry","stress","panic"],
    "loyalty": ["trust","faith","honor","devotion"],
    "mercy": ["kindness","compassion","forgiveness","pity"],
    "dog": ["cat","puppy","animal","pet"],
    "cat": ["dog","kitten","animal","pet"],
    "bread": ["food","butter","loaf","sandwich"],
    "book": ["read","story","novel","library"],
    "river": ["water","stream","ocean","lake"],
    "fear": ["anxiety","worry","panic","dread"],
    "hope": ["dream","wish","faith","future"],
    "joy": ["happiness","pleasure","delight","smile"],
    "thought": ["idea","mind","memory","reason"],
    "memory": ["thought","recall","past","mind"],
    "music": ["sound","song","melody","instrument"],
    "art": ["painting","music","beauty","creation"],
    "law": ["justice","rule","court","right"],
    "war": ["peace","battle","fight","conflict"],
    "peace": ["war","calm","quiet","harmony"],
    "money": ["cash","coin","wealth","currency"],
}

FILLER_POOL = ["cat","dog","car","tree","river","music","book","road",
               "chair","window","garden","cloud","stone","paper","lamp","bird"]
UNRELATED_POOL = ["chair","computer","minute","sky","color","music",
                  "book","weather","engine","river","metal","garden"]
UNRELATED_REPEAT_TOKEN = "table"

WORDLIST_DF.to_csv(DRIVE_BASE/'targets/master_wordlist.csv', index=False)
print(f"saved master wordlist to {DRIVE_BASE/'targets/master_wordlist.csv'}")


In [ ]:
# --- Model panel (open-access, no HF token) ---
MLM_MODELS = [
    ("answerdotai/ModernBERT-base",       149e6),
    ("answerdotai/ModernBERT-large",      395e6),
]
CLM_MODELS = [
    ("Qwen/Qwen2.5-0.5B",                 0.5e9),
    ("Qwen/Qwen2.5-1.5B",                 1.5e9),
    ("Qwen/Qwen2.5-3B",                   3e9),
    ("Qwen/Qwen2.5-7B",                   7e9),
    ("Qwen/Qwen2.5-7B-Instruct",          7e9),
    ("Qwen/Qwen2.5-14B",                 14e9),
    ("HuggingFaceTB/SmolLM2-360M",       360e6),
    ("HuggingFaceTB/SmolLM2-1.7B",       1.7e9),
    ("microsoft/Phi-3.5-mini-instruct",  3.8e9),
    ("allenai/OLMo-2-0425-1B",            1e9),
    ("allenai/OLMo-2-1124-7B",            7e9),
]
ALL_MODELS = MLM_MODELS + CLM_MODELS
MODEL_FAMILY = {mid: "MLM" for mid,_ in MLM_MODELS}
MODEL_FAMILY.update({mid: "CLM" for mid,_ in CLM_MODELS})
MODEL_PARAMS = {mid: p for mid,p in ALL_MODELS}

MECHANISM_MODELS = {
    "answerdotai/ModernBERT-large",
    "Qwen/Qwen2.5-1.5B",
    "Qwen/Qwen2.5-7B",
    "allenai/OLMo-2-1124-7B",
}
ADJACENT_MODELS = {
    "answerdotai/ModernBERT-base",
    "Qwen/Qwen2.5-1.5B",
    "Qwen/Qwen2.5-7B",
}

FRAMES_MLM = [
    ("F0", "The word I keep thinking about is {mask} ."),
    ("F1", "The topic was {mask} ."),
    ("F2", "I want to mention {mask} ."),
    ("F3", "Let me tell you about {mask} ."),
    ("F4", "The most important thing is {mask} ."),
]
FRAMES_CLM = [
    ("F0", "The word I keep thinking about is"),
    ("F1", "The topic was"),
    ("F2", "I want to mention"),
    ("F3", "Let me tell you about"),
    ("F4", "The most important thing is"),
]
ANSWER_FRAMES_CLM = [
    ("A0", "A short document states that the answer is {tgt} . {block} . Therefore, the answer is"),
    ("A1", "The document specifies the answer: {tgt} . {block} . So the answer is"),
    ("A2", "We are told the answer is {tgt} . {block} . Hence the answer is"),
]
ANSWER_FRAMES_MLM = [
    ("A0", "A short document states that the answer is {tgt} . {block} . Therefore, the answer is {mask} ."),
    ("A1", "The document specifies the answer: {tgt} . {block} . So the answer is {mask} ."),
    ("A2", "We are told the answer is {tgt} . {block} . Hence the answer is {mask} ."),
]

NS = [0, 1, 3, 5, 10, 20, 30]
NS_NONZERO = [n for n in NS if n > 0]

def batch_size_for(model_id):
    p = MODEL_PARAMS[model_id]
    if p < 1e9:    return 64
    if p < 2.5e9:  return 32
    if p < 5e9:    return 24
    if p < 8e9:    return 16
    return 8

print(f"panel: {len(ALL_MODELS)} models ({len(MLM_MODELS)} MLM + {len(CLM_MODELS)} CLM)")
for mid, p in ALL_MODELS:
    fam = MODEL_FAMILY[mid]
    extras = []
    if mid in MECHANISM_MODELS: extras.append("mechanism+ablation+application")
    if mid in ADJACENT_MODELS:  extras.append("adjacent")
    print(f"  {fam}  {p/1e6:>7.0f}M  {mid}  {'  '.join(extras) if extras else '(displaced only)'}")


In [ ]:
# --- Tokenizer-level helpers ---

def safe_model_id(mid):
    return mid.replace('/', '__').replace(':','_')

def first_single_token_id(tokenizer, word):
    """Return (token_id, encoded_form) if `word` is single-token under this
    tokenizer, else (None, None)."""
    for form in (" " + word, word):
        try:
            ids = tokenizer.encode(form, add_special_tokens=False)
        except Exception:
            continue
        if len(ids) == 1:
            return ids[0], form
    return None, None

def get_target_ids(tokenizer, words):
    out, forms = {}, {}
    for w in words:
        tid, form = first_single_token_id(tokenizer, w)
        if tid is not None:
            out[w] = tid; forms[w] = form
    return out, forms

def build_per_model_targets(tokenizer, model_id):
    tids, forms = get_target_ids(tokenizer, WORDLIST_DF["word"].tolist())
    df = WORDLIST_DF[WORDLIST_DF["word"].isin(tids)].copy()
    df["token_id"] = df["word"].map(tids)
    df["encoded_form"] = df["word"].map(forms)
    out = DRIVE_BASE/f'targets/targets__{safe_model_id(model_id)}.csv'
    df.to_csv(out, index=False)
    print(f"  {model_id}: {len(df)}/{len(WORDLIST_DF)} single-token-valid -> {out.name}")
    return df

def build_shared_core_targets():
    sets = []
    for mid, _ in ALL_MODELS:
        f = DRIVE_BASE/f'targets/targets__{safe_model_id(mid)}.csv'
        if not f.exists():
            return None
        sets.append(set(pd.read_csv(f)["word"]))
    core = set.intersection(*sets)
    df = WORDLIST_DF[WORDLIST_DF["word"].isin(core)].copy()
    out = DRIVE_BASE/'targets/targets_shared_core.csv'
    df.to_csv(out, index=False)
    print(f"shared core: {len(df)} words -> {out.name}")
    return df


In [ ]:
def load_model(model_id, family):
    """Load model + tokenizer in bfloat16 on CUDA, prefer eager attention so we
    can extract attention weights. Falls back if eager isn't supported."""
    print(f"  loading {model_id} ({family}, bfloat16) ...", flush=True)
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token if tok.eos_token else "[PAD]"
    # Right padding: last real token is at attention_mask.sum() - 1.
    tok.padding_side = "right"
    Cls = AutoModelForMaskedLM if family == "MLM" else AutoModelForCausalLM
    load_kwargs = dict(torch_dtype=torch.bfloat16, trust_remote_code=False)
    try:
        mdl = Cls.from_pretrained(model_id, attn_implementation="eager", **load_kwargs)
    except Exception as e:
        print(f"  [warn] eager attn load failed ({type(e).__name__}); retrying default attn")
        mdl = Cls.from_pretrained(model_id, **load_kwargs)
    mdl = mdl.to("cuda").eval()
    print(f"  loaded in {time.time()-t0:.1f}s  | {sum(p.numel() for p in mdl.parameters())/1e6:.0f}M params", flush=True)
    return tok, mdl

def free_model(tok, mdl):
    try:
        del tok, mdl
    except Exception:
        pass
    gc.collect(); torch.cuda.empty_cache()

def make_input(tokenizer, text, family):
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=True,
                    truncation=True, max_length=512)
    if family == "MLM":
        ids = enc["input_ids"][0]
        mask_id = tokenizer.mask_token_id
        positions = (ids == mask_id).nonzero(as_tuple=True)[0]
        if len(positions) == 0:
            raise ValueError(f"no [MASK] token in: {text!r}")
        slot = int(positions[0])
    else:
        slot = int(enc["input_ids"].shape[1] - 1)
    return enc, slot

def batch_inputs(tokenizer, texts, family):
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    add_special_tokens=True, truncation=True, max_length=512)
    slots = []
    if family == "MLM":
        mask_id = tokenizer.mask_token_id
        for b in range(enc["input_ids"].shape[0]):
            positions = (enc["input_ids"][b] == mask_id).nonzero(as_tuple=True)[0]
            slots.append(int(positions[0]) if len(positions) else -1)
    else:
        for b in range(enc["input_ids"].shape[0]):
            slots.append(int(enc["attention_mask"][b].sum().item()) - 1)
    return enc, slots


In [ ]:
def build_displaced_text(target, n, frame_tmpl, family, mask_str):
    if n == 0:
        return frame_tmpl.format(mask=mask_str) if family == "MLM" else frame_tmpl
    block = " ".join([target] * n)
    rest = frame_tmpl.format(mask=mask_str) if family == "MLM" else frame_tmpl
    return f"{block} . {rest}"

def build_adjacent_text(target, n, family, mask_str):
    if family == "MLM":
        if n == 0: return f"{mask_str} ."
        block = " ".join([target] * n)
        return f"{block} {mask_str} ."
    else:
        if n == 0: return ""
        return " ".join([target] * n)

def build_ablation_text(target, condition, frame_tmpl, family, mask_str, rng, sem_pool):
    rest = frame_tmpl.format(mask=mask_str) if family == "MLM" else frame_tmpl
    if condition == "full_repeat_N30":
        return " ".join([target]*30) + " . " + rest, 30
    if condition == "truncate_to_N3":
        return " ".join([target]*3) + " . " + rest, 3
    if condition == "delete_repeat_block":
        return rest, 0
    if condition == "random_filler_same_length":
        fillers = [rng.choice(FILLER_POOL) for _ in range(30)]
        return " ".join(fillers) + " . " + rest, 0
    if condition == "unique_semantic_filler":
        if not sem_pool: return None, None
        fillers = [rng.choice(sem_pool) for _ in range(30)]
        return " ".join(fillers) + " . " + rest, 0
    if condition == "repeat_unrelated_N30":
        return " ".join([UNRELATED_REPEAT_TOKEN]*30) + " . " + rest, 30
    return None, None

def build_application_text(target, n, condition, frame_tmpl, family, mask_str, rng, sem_pool):
    if n == 0:
        block = ""
    elif condition == "target_repeat":
        block = " ".join([target] * n)
    elif condition == "unrelated_repeat":
        block = " ".join([UNRELATED_REPEAT_TOKEN] * n)
    elif condition == "random_repeat":
        block = " ".join([rng.choice(FILLER_POOL) for _ in range(n)])
    elif condition == "semantic_repeat":
        if not sem_pool: return None
        block = " ".join([rng.choice(sem_pool) for _ in range(n)])
    else:
        return None
    if family == "MLM":
        return frame_tmpl.format(tgt=target, block=block, mask=mask_str)
    return frame_tmpl.format(tgt=target, block=block)


In [ ]:
@torch.no_grad()
def score_batch(tokenizer, model, family, texts, target_ids, sem_field_ids, unrelated_ids):
    enc, slots = batch_inputs(tokenizer, texts, family)
    enc = {k: v.to("cuda") for k, v in enc.items()}
    out = model(**enc)
    logits = out.logits.float()
    rows = []
    for b in range(logits.shape[0]):
        slot = slots[b]
        if slot < 0:
            rows.append({"target_prob": float("nan"), "target_rank": -1,
                         "entropy": float("nan"),
                         "sum_synonym_prob": float("nan"),
                         "sum_unrelated_pool_prob": float("nan")})
            continue
        lg = logits[b, slot]
        probs = F.softmax(lg, dim=-1)
        tid = target_ids[b]
        if tid is not None:
            tp = float(probs[tid].item())
            rank = int((probs > probs[tid]).sum().item()) + 1
        else:
            tp = float("nan"); rank = -1
        ent = float(-(probs.clamp_min(1e-12).log() * probs).sum().item())
        sf_ids = sem_field_ids[b]
        sf_mass = float(probs[sf_ids].sum().item()) if sf_ids else float("nan")
        un_mass = float(probs[unrelated_ids].sum().item()) if unrelated_ids else float("nan")
        rows.append({
            "target_prob": tp, "target_rank": rank, "entropy": ent,
            "sum_synonym_prob": sf_mass, "sum_unrelated_pool_prob": un_mass,
        })
    return rows

def chunked(seq, k):
    for i in range(0, len(seq), k):
        yield seq[i:i+k]

def build_sem_field_ids(tokenizer, target_ids):
    out = {}
    for w, neighbours in SEM_FIELD.items():
        if w not in target_ids: continue
        ids = []
        for nb in neighbours:
            if nb == w: continue
            tid, _ = first_single_token_id(tokenizer, nb)
            if tid is not None:
                ids.append(tid)
        if ids: out[w] = ids
    return out

def build_unrelated_ids(tokenizer):
    ids = []
    for u in UNRELATED_POOL:
        tid, _ = first_single_token_id(tokenizer, u)
        if tid is not None:
            ids.append(tid)
    return ids


In [ ]:
def run_displaced(model_id, tokenizer, model, family, targets_df, batch_size):
    mask_str = tokenizer.mask_token if family == "MLM" else None
    target_ids = dict(zip(targets_df["word"], targets_df["token_id"].astype(int)))
    sem_ids = build_sem_field_ids(tokenizer, target_ids)
    un_ids = build_unrelated_ids(tokenizer)

    items = []
    frames = FRAMES_MLM if family == "MLM" else FRAMES_CLM
    for fid, ftmpl in frames:
        for _, w in targets_df.iterrows():
            for n in NS:
                items.append({
                    "model": model_id, "family": family,
                    "frame_id": fid, "frame_text": ftmpl,
                    "target_word": w["word"],
                    "category": w["category"],
                    "concreteness_group": w["concreteness_group"],
                    "frequency_group": w["frequency_group"],
                    "semantic_class": w["semantic_class"],
                    "zipf_frequency": float(w["zipf_frequency"]),
                    "N": n,
                    "text": build_displaced_text(w["word"], n, ftmpl, family, mask_str),
                })

    print(f"  displaced: {len(items)} items, batch={batch_size}")
    rows = []
    t0 = time.time()
    for batch in chunked(items, batch_size):
        texts = [it["text"] for it in batch]
        tids  = [target_ids[it["target_word"]] for it in batch]
        sfids = [sem_ids.get(it["target_word"]) for it in batch]
        scores = score_batch(tokenizer, model, family, texts, tids, sfids, un_ids)
        for it, sc in zip(batch, scores):
            r = {**it, **sc}; r.pop("text", None)
            rows.append(r)
    print(f"  displaced done in {time.time()-t0:.0f}s")
    df = pd.DataFrame(rows)
    out = DRIVE_BASE/f'displaced/displaced__{safe_model_id(model_id)}.parquet'
    df.to_parquet(out, index=False)
    print(f"  saved {out}")
    return df

def run_adjacent(model_id, tokenizer, model, family, targets_df, batch_size):
    mask_str = tokenizer.mask_token if family == "MLM" else None
    target_ids = dict(zip(targets_df["word"], targets_df["token_id"].astype(int)))
    sem_ids = build_sem_field_ids(tokenizer, target_ids)
    un_ids = build_unrelated_ids(tokenizer)
    items = []
    for _, w in targets_df.iterrows():
        for n in NS:
            text = build_adjacent_text(w["word"], n, family, mask_str)
            if text == "":
                continue
            items.append({
                "model": model_id, "family": family,
                "target_word": w["word"], "N": n,
                "zipf_frequency": float(w["zipf_frequency"]),
                "category": w["category"],
                "text": text,
            })
    print(f"  adjacent: {len(items)} items, batch={batch_size}")
    rows = []
    for batch in chunked(items, batch_size):
        texts = [it["text"] for it in batch]
        tids  = [target_ids[it["target_word"]] for it in batch]
        sfids = [sem_ids.get(it["target_word"]) for it in batch]
        scores = score_batch(tokenizer, model, family, texts, tids, sfids, un_ids)
        for it, sc in zip(batch, scores):
            r = {**it, **sc}; r.pop("text", None)
            rows.append(r)
    df = pd.DataFrame(rows)
    out = DRIVE_BASE/f'adjacent/adjacent__{safe_model_id(model_id)}.parquet'
    df.to_parquet(out, index=False)
    print(f"  saved {out}")
    return df

def run_ablation(model_id, tokenizer, model, family, targets_df, batch_size, seed=42):
    mask_str = tokenizer.mask_token if family == "MLM" else None
    target_ids = dict(zip(targets_df["word"], targets_df["token_id"].astype(int)))
    sem_ids = build_sem_field_ids(tokenizer, target_ids)
    un_ids = build_unrelated_ids(tokenizer)
    sem_word_pool = {}
    for w, nbs in SEM_FIELD.items():
        if w not in target_ids: continue
        sem_word_pool[w] = [nb for nb in nbs if first_single_token_id(tokenizer, nb)[0] is not None]
    rng = random.Random(seed)
    fid, ftmpl = (FRAMES_MLM if family == "MLM" else FRAMES_CLM)[0]
    conditions = ["full_repeat_N30","truncate_to_N3","delete_repeat_block",
                  "random_filler_same_length","unique_semantic_filler",
                  "repeat_unrelated_N30"]
    items = []
    for _, w in targets_df.iterrows():
        word = w["word"]
        sem_pool = sem_word_pool.get(word, [])
        for cond in conditions:
            text, N = build_ablation_text(word, cond, ftmpl, family, mask_str, rng, sem_pool)
            if text is None: continue
            items.append({
                "model": model_id, "family": family,
                "frame_id": fid, "condition": cond, "N": N,
                "target_word": word,
                "category": w["category"], "zipf_frequency": float(w["zipf_frequency"]),
                "text": text,
            })
    print(f"  ablation: {len(items)} items, batch={batch_size}")
    rows = []
    for batch in chunked(items, batch_size):
        texts = [it["text"] for it in batch]
        tids  = [target_ids[it["target_word"]] for it in batch]
        sfids = [sem_ids.get(it["target_word"]) for it in batch]
        scores = score_batch(tokenizer, model, family, texts, tids, sfids, un_ids)
        for it, sc in zip(batch, scores):
            r = {**it, **sc}; r.pop("text", None)
            rows.append(r)
    df = pd.DataFrame(rows)
    out = DRIVE_BASE/f'ablation/ablation__{safe_model_id(model_id)}.parquet'
    df.to_parquet(out, index=False)
    print(f"  saved {out}")
    return df


## Frame-pragmatics ablation (mechanism subset)

Holds the displaced readout position and the repeated-target block fixed and varies only the frame's pragmatic **class** (topic-introducer / neutral / anaphoric / novelty) on the four mechanism models. Reuses the displaced scoring path; writes per-model parquet to `frame_pragmatics/` and an aggregated `summary/frame_pragmatics_summary.csv` (per-word drop, 500-resample bootstrap, seed 42).

In [ ]:
# --- Frame-pragmatics ablation (four mechanism models) ---
# Same displaced readout, same repeated-target block; only the frame's
# pragmatic CLASS varies. Frames are stored bare; MLM appends " {mask} .".
FRAMEPRAGM = {
    "ORIG":      [("F0", "The word I keep thinking about is"),
                  ("F2", "I want to mention")],
    "NEUTRAL":   [("NE0", "I saw a"), ("NE1", "There was a"), ("NE2", "Look, a")],
    "ANAPHORIC": [("AN0", "That word again:"),
                  ("AN1", "The word repeated above is"),
                  ("AN2", "To say it once more, the word is")],
    "NOVELTY":   [("NV0", "On a completely different note, the word is"),
                  ("NV1", "Changing the subject, the new word is"),
                  ("NV2", "Now for something different, the word is")],
}

def _pragm_tmpl(ftext, family):
    return (ftext + " {mask} .") if family == "MLM" else ftext

def run_frame_pragmatics(model_id, tokenizer, model, family, targets_df, batch_size):
    mask_str = tokenizer.mask_token if family == "MLM" else None
    target_ids = dict(zip(targets_df["word"], targets_df["token_id"].astype(int)))
    sem_ids = build_sem_field_ids(tokenizer, target_ids)
    un_ids = build_unrelated_ids(tokenizer)
    items = []
    for cls, frames in FRAMEPRAGM.items():
        for fid, ftext in frames:
            ftmpl = _pragm_tmpl(ftext, family)
            for _, w in targets_df.iterrows():
                for n in NS:
                    items.append({
                        "model": model_id, "family": family,
                        "frame_class": cls, "frame_id": fid, "frame_text": ftext,
                        "target_word": w["word"],
                        "category": w["category"],
                        "concreteness_group": w["concreteness_group"],
                        "frequency_group": w["frequency_group"],
                        "semantic_class": w["semantic_class"],
                        "zipf_frequency": float(w["zipf_frequency"]),
                        "N": n,
                        "text": build_displaced_text(w["word"], n, ftmpl, family, mask_str),
                    })
    print(f"  frame_pragmatics: {len(items)} items, batch={batch_size}")
    rows = []
    t0 = time.time()
    for batch in chunked(items, batch_size):
        texts = [it["text"] for it in batch]
        tids  = [target_ids[it["target_word"]] for it in batch]
        sfids = [sem_ids.get(it["target_word"]) for it in batch]
        scores = score_batch(tokenizer, model, family, texts, tids, sfids, un_ids)
        for it, sc in zip(batch, scores):
            r = {**it, **sc}; r.pop("text", None)
            rows.append(r)
    print(f"  frame_pragmatics done in {time.time()-t0:.0f}s")
    df = pd.DataFrame(rows)
    (DRIVE_BASE/"frame_pragmatics").mkdir(parents=True, exist_ok=True)
    out = DRIVE_BASE/f'frame_pragmatics/frame_pragmatics__{safe_model_id(model_id)}.parquet'
    df.to_parquet(out, index=False)
    print(f"  saved {out}")
    return df

def summarize_frame_pragmatics(seed=42, n_boot=500):
    import glob
    files = sorted(glob.glob(str(DRIVE_BASE/"frame_pragmatics/frame_pragmatics__*.parquet")))
    if not files:
        print("  no frame_pragmatics files; skip summary"); return
    fp = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    fp.to_parquet(DRIVE_BASE/"summary"/"frame_pragmatics_all.parquet", index=False)
    rng = np.random.default_rng(seed)
    ns_nz = [n for n in NS if n > 0]
    rows = []
    for (mid, cls), g in fp.groupby(["model", "frame_class"]):
        piv = g.pivot_table(index="target_word", columns="N", values="target_prob", aggfunc="mean")
        cols = [n for n in ns_nz if n in piv.columns]
        peak = piv[cols].max(axis=1); p30 = piv[30]
        d = ((peak - p30) / peak)[peak > 0].values
        boots = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
        mc = g.groupby("N")["target_prob"].mean(); mcs = mc[[n for n in ns_nz if n in mc.index]]
        pk = int(mcs.idxmax())
        rows.append({"model": mid, "frame_class": cls,
                     "peak_N_mean_curve": pk, "peak_P_mean_curve": float(mcs.max()),
                     "P_N30_mean_curve": float(mc[30]), "n_words": int(len(d)),
                     "drop_mean": float(d.mean()),
                     "drop_ci_lo": float(np.percentile(boots, 2.5)),
                     "drop_ci_hi": float(np.percentile(boots, 97.5)),
                     "mean_curve_inverts": bool(pk < 30)})
    pd.DataFrame(rows).to_csv(DRIVE_BASE/"summary"/"frame_pragmatics_summary.csv", index=False)
    print(f"  saved frame_pragmatics_summary.csv ({len(rows)} rows)")


In [ ]:
@torch.no_grad()
def attention_at_slot(tokenizer, model, family, text, target_token_id):
    enc, slot = make_input(tokenizer, text, family)
    enc = {k: v.to("cuda") for k, v in enc.items()}
    out = model(**enc, output_attentions=True)
    a = out.attentions[-1][0].float()
    a_slot = a[:, slot, :].mean(dim=0)
    ids = enc["input_ids"][0]
    tgt_positions = (ids == target_token_id).nonzero(as_tuple=True)[0].tolist()
    n_target_pos = len(tgt_positions)
    if n_target_pos == 0:
        return float("nan"), float("nan"), 0
    attn_total = float(a_slot[tgt_positions].sum().item())
    return attn_total / n_target_pos, attn_total, n_target_pos

def get_final_norm_and_head(model):
    final_norm = None
    for path in ["model.norm", "model.final_layernorm", "model.ln_f", "transformer.ln_f"]:
        cur = model
        ok = True
        for part in path.split("."):
            if not hasattr(cur, part): ok = False; break
            cur = getattr(cur, part)
        if ok:
            final_norm = cur; break
    lm_head = model.get_output_embeddings()
    return final_norm, lm_head

@torch.no_grad()
def slot_state_metrics(tokenizer, model, family, text, target_token_id):
    enc, slot = make_input(tokenizer, text, family)
    enc = {k: v.to("cuda") for k, v in enc.items()}
    out = model(**enc, output_hidden_states=True)
    last_logits = out.logits.float()[0, slot]
    target_prob = float(F.softmax(last_logits, dim=-1)[target_token_id].item())
    if family == "MLM":
        h = out.hidden_states[-1][0, slot].float()
        emb = model.get_input_embeddings().weight.float()
        cos = float(F.cosine_similarity(h.unsqueeze(0),
                                        emb[target_token_id].unsqueeze(0)).item())
        return {"cos_mask_to_target_embed": cos, "target_prob": target_prob}
    else:
        final_norm, lm_head = get_final_norm_and_head(model)
        layer_logp = []
        for hs in out.hidden_states:
            h = hs[0, slot]
            try:
                hn = final_norm(h.unsqueeze(0)).squeeze(0)
            except Exception:
                hn = h
            lg = lm_head(hn.to(lm_head.weight.dtype)).float()
            lp = float(F.log_softmax(lg, dim=-1)[target_token_id].item())
            layer_logp.append(lp)
        return {"layer_logit_lens_logp": layer_logp, "target_prob": target_prob}

def run_mechanism_last_layer(model_id, tokenizer, model, family, targets_df, n_words=10):
    mask_str = tokenizer.mask_token if family == "MLM" else None
    fid, ftmpl = (FRAMES_MLM if family == "MLM" else FRAMES_CLM)[0]
    sample = targets_df.sample(min(n_words, len(targets_df)), random_state=0).reset_index(drop=True)
    target_ids = dict(zip(sample["word"], sample["token_id"].astype(int)))
    rows = []
    for _, w in sample.iterrows():
        for n in NS:
            text = build_displaced_text(w["word"], n, ftmpl, family, mask_str)
            try:
                per_tok, total, n_pos = attention_at_slot(
                    tokenizer, model, family, text, target_ids[w["word"]])
            except Exception:
                per_tok, total, n_pos = float("nan"), float("nan"), 0
            rows.append({
                "model": model_id, "family": family, "frame_id": fid,
                "target_word": w["word"], "N": n,
                "attn_per_target_token": per_tok,
                "attn_total_to_target_block": total,
                "n_target_positions": n_pos,
            })
    df = pd.DataFrame(rows)
    out = DRIVE_BASE/f'mechanism/attention_last_layer__{safe_model_id(model_id)}.csv'
    df.to_csv(out, index=False)
    print(f"  mechanism (attention) saved {out}")

    sample23 = targets_df.sample(min(23, len(targets_df)), random_state=1).reset_index(drop=True)
    target_ids23 = dict(zip(sample23["word"], sample23["token_id"].astype(int)))
    rows = []
    for _, w in sample23.iterrows():
        for n in NS:
            text = build_displaced_text(w["word"], n, ftmpl, family, mask_str)
            try:
                m = slot_state_metrics(tokenizer, model, family, text, target_ids23[w["word"]])
            except Exception:
                m = {"target_prob": float("nan")}
            row = {"model": model_id, "family": family, "frame_id": fid,
                   "target_word": w["word"], "N": n, "target_prob": m.get("target_prob", float("nan"))}
            if family == "MLM":
                row["cos_mask_to_target_embed"] = m.get("cos_mask_to_target_embed", float("nan"))
            rows.append(row)
    df = pd.DataFrame(rows)
    out = DRIVE_BASE/f'mechanism/slot_state__{safe_model_id(model_id)}.csv'
    df.to_csv(out, index=False)
    print(f"  mechanism (slot state) saved {out}")


In [ ]:
def run_layerwise(model_id, tokenizer, model, family, targets_df, n_words=50):
    mask_str = tokenizer.mask_token if family == "MLM" else None
    fid, ftmpl = (FRAMES_MLM if family == "MLM" else FRAMES_CLM)[0]
    sample = targets_df.sample(min(n_words, len(targets_df)), random_state=2).reset_index(drop=True)
    target_ids = dict(zip(sample["word"], sample["token_id"].astype(int)))
    rows = []
    final_norm = None; lm_head = None
    if family == "CLM":
        final_norm, lm_head = get_final_norm_and_head(model)
    for _, w in sample.iterrows():
        for n in NS:
            text = build_displaced_text(w["word"], n, ftmpl, family, mask_str)
            try:
                enc, slot = make_input(tokenizer, text, family)
                enc = {k: v.to("cuda") for k, v in enc.items()}
                with torch.no_grad():
                    out = model(**enc, output_attentions=True, output_hidden_states=True)
                ids = enc["input_ids"][0]
                tgt_pos = (ids == target_ids[w["word"]]).nonzero(as_tuple=True)[0].tolist()
                n_pos = len(tgt_pos)
                # per-layer attention rows (layers 1..L)
                for layer_i, attn in enumerate(out.attentions):
                    a_slot = attn[0].float().mean(dim=0)[slot]
                    if n_pos > 0:
                        total = float(a_slot[tgt_pos].sum().item())
                        per_tok = total / n_pos
                    else:
                        per_tok, total = float("nan"), float("nan")
                    rows.append({
                        "model": model_id, "family": family, "frame_id": fid,
                        "target_word": w["word"], "N": n, "layer": layer_i + 1,
                        "n_layers": len(out.attentions),
                        "metric": "attention",
                        "attn_per_target_token": per_tok,
                        "attn_total_to_target_block": total,
                        "n_target_positions": n_pos,
                        "logit_lens_logp": float("nan"),
                    })
                # CLM-only logit-lens rows (layers 0..L)
                if family == "CLM" and lm_head is not None:
                    tid = target_ids[w["word"]]
                    for li, hs in enumerate(out.hidden_states):
                        h = hs[0, slot]
                        try:
                            hn = final_norm(h.unsqueeze(0)).squeeze(0)
                        except Exception:
                            hn = h
                        lg = lm_head(hn.to(lm_head.weight.dtype)).float()
                        lp = float(F.log_softmax(lg, dim=-1)[tid].item())
                        rows.append({
                            "model": model_id, "family": family, "frame_id": fid,
                            "target_word": w["word"], "N": n, "layer": li,
                            "n_layers": len(out.hidden_states),
                            "metric": "logit_lens",
                            "attn_per_target_token": float("nan"),
                            "attn_total_to_target_block": float("nan"),
                            "n_target_positions": n_pos,
                            "logit_lens_logp": lp,
                        })
                del out
            except Exception:
                pass
    df = pd.DataFrame(rows)
    out = DRIVE_BASE/f'layerwise/layerwise__{safe_model_id(model_id)}.parquet'
    df.to_parquet(out, index=False)
    print(f"  layerwise saved {out}")

def run_application(model_id, tokenizer, model, family, targets_df, batch_size, n_words=50, seed=0):
    mask_str = tokenizer.mask_token if family == "MLM" else None
    target_ids = dict(zip(targets_df["word"], targets_df["token_id"].astype(int)))
    sem_ids = build_sem_field_ids(tokenizer, target_ids)
    un_ids = build_unrelated_ids(tokenizer)
    sem_word_pool = {}
    for w, nbs in SEM_FIELD.items():
        if w not in target_ids: continue
        sem_word_pool[w] = [nb for nb in nbs if first_single_token_id(tokenizer, nb)[0] is not None]
    sample = targets_df.sample(min(n_words, len(targets_df)), random_state=seed).reset_index(drop=True)
    rng = random.Random(seed)
    frames = ANSWER_FRAMES_MLM if family == "MLM" else ANSWER_FRAMES_CLM
    conditions = ["target_repeat","random_repeat","unrelated_repeat","semantic_repeat"]
    items = []
    for fid, ftmpl in frames:
        for _, w in sample.iterrows():
            word = w["word"]
            sem_pool = sem_word_pool.get(word, [])
            for cond in conditions:
                for n in NS_NONZERO:
                    text = build_application_text(word, n, cond, ftmpl, family, mask_str, rng, sem_pool)
                    if text is None: continue
                    items.append({
                        "model": model_id, "family": family,
                        "template_id": fid, "condition": cond, "N": n,
                        "target_word": word,
                        "category": w["category"], "zipf_frequency": float(w["zipf_frequency"]),
                        "text": text,
                    })
    print(f"  application: {len(items)} items, batch={batch_size}")
    rows = []
    for batch in chunked(items, batch_size):
        texts = [it["text"] for it in batch]
        tids  = [target_ids[it["target_word"]] for it in batch]
        sfids = [sem_ids.get(it["target_word"]) for it in batch]
        scores = score_batch(tokenizer, model, family, texts, tids, sfids, un_ids)
        for it, sc in zip(batch, scores):
            r = {**it, **sc}; r.pop("text", None)
            rows.append(r)
    df = pd.DataFrame(rows)
    out = DRIVE_BASE/f'application/application__{safe_model_id(model_id)}.parquet'
    df.to_parquet(out, index=False)
    print(f"  saved {out}")
    return df


In [ ]:
MANIFEST_PATH = DRIVE_BASE/"summary"/"manifest.json"

def load_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {"completed": {}, "started_at": datetime.now(timezone.utc).isoformat()}

def save_manifest(m):
    MANIFEST_PATH.write_text(json.dumps(m, indent=2, sort_keys=True))

def log_line(msg):
    line = f"[{datetime.now(timezone.utc).isoformat()}] {msg}"
    print(line, flush=True)
    with open(DRIVE_BASE/"summary"/"logs"/"run.log", "a") as f:
        f.write(line + "\n")

def run_one_model(model_id):
    family = MODEL_FAMILY[model_id]
    bs = batch_size_for(model_id)
    log_line(f"=== {model_id} ({family}, ~{MODEL_PARAMS[model_id]/1e9:.2f}B, bs={bs}) ===")
    tok = mdl = None
    t0 = time.time()
    try:
        tok, mdl = load_model(model_id, family)
        targets = build_per_model_targets(tok, model_id)
        if len(targets) < 20:
            log_line(f"  WARN only {len(targets)} single-token-valid words; skipping experiments")
            free_model(tok, mdl); return False

        run_displaced(model_id, tok, mdl, family, targets, bs)
        if model_id in ADJACENT_MODELS:
            run_adjacent(model_id, tok, mdl, family, targets, bs)
        if model_id in MECHANISM_MODELS:
            run_ablation(model_id, tok, mdl, family, targets, bs)
            run_mechanism_last_layer(model_id, tok, mdl, family, targets)
            run_layerwise(model_id, tok, mdl, family, targets)
            run_application(model_id, tok, mdl, family, targets, bs)
            run_frame_pragmatics(model_id, tok, mdl, family, targets, bs)

        free_model(tok, mdl)
        log_line(f"  done in {(time.time()-t0)/60:.1f} min")
        return True
    except Exception as e:
        log_line(f"  FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()
        try: free_model(tok, mdl)
        except Exception: pass
        gc.collect(); torch.cuda.empty_cache()
        return False

manifest = load_manifest()
for mid, _ in ALL_MODELS:
    if mid in manifest["completed"]:
        log_line(f"skip {mid} (already done at {manifest['completed'][mid]})")
        continue
    ok = run_one_model(mid)
    if ok:
        manifest["completed"][mid] = datetime.now(timezone.utc).isoformat()
    save_manifest(manifest)

log_line("=== panel run finished ===")
log_line(f"completed: {len(manifest['completed'])}/{len(ALL_MODELS)}")
build_shared_core_targets()
summarize_frame_pragmatics()


In [ ]:
import glob

displaced_files = sorted(glob.glob(str(DRIVE_BASE/"displaced/displaced__*.parquet")))
print(f"merging {len(displaced_files)} displaced files")
disp = pd.concat([pd.read_parquet(f) for f in displaced_files], ignore_index=True) if displaced_files else pd.DataFrame()
if len(disp):
    disp.to_parquet(DRIVE_BASE/"summary"/"displaced_all.parquet", index=False)
    print(f"saved displaced_all.parquet  shape={disp.shape}")

def per_word_drop_table(disp_df):
    rows = []
    for (model, frame, word), g in disp_df.groupby(["model","frame_id","target_word"]):
        g = g.set_index("N")["target_prob"].sort_index()
        if not all(n in g.index for n in NS_NONZERO):
            continue
        sub = g.loc[NS_NONZERO]
        # all-NaN safeguard: some (model, frame, word) groups can have
        # NaN target_prob across all N (e.g., when the slot couldn't be
        # resolved); skip those rather than letting idxmax raise.
        if not sub.notna().any():
            continue
        peak = float(sub.max(skipna=True))
        if not np.isfinite(peak) or peak <= 0:
            continue
        if 30 not in g.index or pd.isna(g.loc[30]):
            continue
        peak_n = int(sub.idxmax(skipna=True))
        p30 = float(g.loc[30])
        drop = (peak - p30) / peak
        rows.append({"model": model, "frame_id": frame, "target_word": word,
                     "peak_N": peak_n, "peak_P": peak, "P_N30": p30, "drop": drop})
    return pd.DataFrame(rows)

if len(disp):
    drop_df = per_word_drop_table(disp)
    drop_df.to_parquet(DRIVE_BASE/"summary"/"per_word_drops.parquet", index=False)
    print(f"saved per_word_drops.parquet  shape={drop_df.shape}")

    def model_frame_summary(drop_df, disp_df):
        rows = []
        for (model, frame), g in drop_df.groupby(["model","frame_id"]):
            rng = np.random.default_rng(42)
            vals = g["drop"].dropna().values
            if len(vals) == 0:
                continue
            boots = [float(rng.choice(vals, size=len(vals), replace=True).mean()) for _ in range(500)]
            lo, hi = float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
            sub = disp_df[(disp_df.model==model) & (disp_df.frame_id==frame)]
            mc = sub.groupby("N")["target_prob"].mean().sort_index()
            nz = mc.loc[[n for n in NS_NONZERO if n in mc.index]].dropna()
            if len(nz) == 0 or 30 not in mc.index or pd.isna(mc.loc[30]):
                continue
            peak_meanN = int(nz.idxmax())
            peak_meanP = float(nz.max())
            rows.append({"model": model, "frame_id": frame,
                         "peak_N_mean_curve": peak_meanN,
                         "peak_P_mean_curve": peak_meanP,
                         "P_N30_mean_curve": float(mc.loc[30]),
                         "n_words": int(len(vals)),
                         "drop_mean": float(vals.mean()),
                         "drop_ci_lo": lo, "drop_ci_hi": hi})
        return pd.DataFrame(rows)

    mf = model_frame_summary(drop_df, disp)
    mf.to_csv(DRIVE_BASE/"summary"/"model_frame_summary.csv", index=False)
    print(f"saved model_frame_summary.csv  shape={mf.shape}")
    print(mf.head(10))


In [ ]:
import statsmodels.api as sm

shared_csv = DRIVE_BASE/"targets/targets_shared_core.csv"
if shared_csv.exists() and len(disp):
    shared = pd.read_csv(shared_csv)
    sub = disp[(disp["target_word"].isin(shared["word"])) & (disp["N"] >= 1)].copy()
    # Drop rows where target_prob is NaN/non-finite/non-positive — log()
    # would propagate NaN through the entire OLS fit otherwise.
    before = len(sub)
    sub = sub[sub["target_prob"].notna() & (sub["target_prob"] > 0)]
    sub["log_p"] = np.log(sub["target_prob"].astype(float))
    sub = sub[np.isfinite(sub["log_p"])]
    sub = sub.dropna(subset=["zipf_frequency","model","frame_id","category","N"])
    print(f"OLS rows after dropping NaN: {before} -> {len(sub)}")
    sub["N2"] = sub["N"]**2
    # family_CLM is collinear with model fixed effects, so it is omitted.
    Xd = pd.get_dummies(sub[["model","frame_id","category"]], drop_first=True).astype(float)
    Xd["N"] = sub["N"].astype(float)
    Xd["N2"] = sub["N2"].astype(float)
    Xd["zipf"] = sub["zipf_frequency"].astype(float)
    Xd = sm.add_constant(Xd)
    y = sub["log_p"].astype(float)
    assert not y.isna().any() and not Xd.isna().any().any(), "OLS inputs still have NaN"
    res = sm.OLS(y, Xd).fit(cov_type="HC0")
    # res.summary() runs a joint F-test that can raise with HC0 + many
    # dummies; use summary2() (or fall back to the coef table) to skip it.
    try:
        summary_str = res.summary2().as_text()
    except Exception:
        coef_df = pd.DataFrame({"coef": res.params, "se": res.bse,
                                "t": res.tvalues, "p": res.pvalues})
        summary_str = (f"n={int(res.nobs)}  R^2={res.rsquared:.4f}  "
                       f"adjR^2={res.rsquared_adj:.4f}\n\n{coef_df.to_string()}")
    (DRIVE_BASE/"summary"/"ols_quadratic_shared_core.txt").write_text(summary_str)
    print(f"OLS shared-core: n={int(res.nobs)}, R^2={res.rsquared:.4f}")
    print(summary_str[:2000])
    print()
    for k in ["N","N2","zipf"]:
        if k in res.params.index:
            print(f"  beta_{k:5s} = {res.params[k]:+.5f}   "
                  f"SE={res.bse[k]:.5f}   t={res.tvalues[k]:+.2f}   "
                  f"p={res.pvalues[k]:.3g}")
else:
    print("shared core not yet built or no displaced data; skipping cross-model OLS")


In [ ]:
manifest = load_manifest()
print(f"=== final summary ===")
print(f"output base: {DRIVE_BASE}")
print(f"completed: {len(manifest['completed'])}/{len(ALL_MODELS)}")
for mid, _ in ALL_MODELS:
    state = "DONE" if mid in manifest["completed"] else "MISSING"
    print(f"  {state}  {mid}")
print()
print("file inventory:")
for sub in ['targets','displaced','adjacent','ablation','mechanism','layerwise','application']:
    files = sorted((DRIVE_BASE/sub).glob('*'))
    if files:
        print(f"  {sub}/ ({len(files)} files)")
        for f in files[:3]:
            print(f"    {f.name}  ({f.stat().st_size/1024:.0f} KB)")
        if len(files) > 3:
            print(f"    ... and {len(files)-3} more")


## §2 Multilingual panel

4 models × 4 languages cross-lingual scope check (XLM-R-base,
XLM-R-large, Qwen2.5-1.5B, Qwen2.5-7B on es / zh / de / fr; XLM-R
Chinese excluded by tokenizer filter, leaving 14 (model, language)
combinations × 3 frames = 42 evaluated cells).

In [ ]:
# Hand-curated multilingual single-token noun lists, balanced across
# concrete/abstract and high/low Zipf where possible. Numbers chosen to
# parallel the legacy multilingual run (n ≈ 50 per language).

WORDLISTS = {
    "es": [
        "agua","casa","tiempo","mundo","vida","mano","mujer","hombre","niño","libro",
        "ciudad","pueblo","persona","amor","trabajo","amigo","familia","historia","cuerpo","cabeza",
        "padre","madre","hijo","hija","puerta","ventana","pared","calle","camino","puente",
        "piedra","árbol","flor","sol","luna","estrella","fuego","río","mar","montaña",
        "verdad","mentira","alma","corazón","muerte","sueño","espíritu","libertad","justicia","paz",
    ],
    "zh": [
        "水","时间","人","年","月","日","年","事","物","车",
        "山","河","海","风","云","雨","雪","花","树","草",
        "心","头","手","脚","眼","口","门","窗","路","桥",
        "书","纸","笔","字","词","话","梦","爱","真","假",
        "光","暗","声","影","动","静","生","死","新","老",
    ],
    "de": [
        "Wasser","Haus","Zeit","Welt","Leben","Frau","Mann","Kind","Buch","Stadt",
        "Land","Familie","Liebe","Arbeit","Freund","Geschichte","Körper","Kopf","Vater","Mutter",
        "Sohn","Tochter","Tür","Fenster","Wand","Straße","Weg","Brücke","Stein","Baum",
        "Blume","Sonne","Mond","Stern","Feuer","Fluss","Meer","Berg","Wahrheit","Lüge",
        "Seele","Herz","Tod","Traum","Geist","Freiheit","Gerechtigkeit","Frieden","Hoffnung","Krieg",
    ],
    "fr": [
        "eau","maison","temps","monde","vie","femme","homme","enfant","livre","ville",
        "pays","famille","amour","travail","ami","histoire","corps","tête","père","mère",
        "fils","fille","porte","fenêtre","mur","rue","chemin","pont","pierre","arbre",
        "fleur","soleil","lune","étoile","feu","rivière","mer","montagne","vérité","mensonge",
        "âme","coeur","mort","rêve","esprit","liberté","justice","paix","espoir","guerre",
    ],
}
# Three frames per language, hand-translated to idiomatic equivalents
# of the English F0/F1/F2 templates.
FRAMES_MLM = {
    "es": [("F0","La palabra que sigo pensando es {mask} ."),
           ("F1","El tema era {mask} ."),
           ("F2","Quiero mencionar {mask} .")],
    "zh": [("F0","我一直在想的词是 {mask} 。"),
           ("F1","主题是 {mask} 。"),
           ("F2","我想提到 {mask} 。")],
    "de": [("F0","Das Wort, an das ich immer wieder denke, ist {mask} ."),
           ("F1","Das Thema war {mask} ."),
           ("F2","Ich möchte {mask} erwähnen .")],
    "fr": [("F0","Le mot auquel je continue de penser est {mask} ."),
           ("F1","Le sujet était {mask} ."),
           ("F2","Je veux mentionner {mask} .")],
}
FRAMES_CLM = {lang: [(fid, t.replace(" {mask} .","").replace(" {mask}。","")) for fid, t in flist]
              for lang, flist in FRAMES_MLM.items()}

NS = [0, 1, 3, 5, 10, 20, 30]
NS_NONZERO = [n for n in NS if n > 0]

print("languages:", list(WORDLISTS.keys()))
print("words per language:", {k: len(v) for k, v in WORDLISTS.items()})


In [ ]:
ML_MODELS = [
    # MLM (multilingual encoders)
    ("FacebookAI/xlm-roberta-base",   "XLM-R-base",   "MLM",  279e6),
    ("FacebookAI/xlm-roberta-large",  "XLM-R-large",  "MLM",  561e6),
    # CLM (multilingual decoders)
    ("Qwen/Qwen2.5-1.5B",             "Qwen2.5-1.5B", "CLM",  1.5e9),
    ("Qwen/Qwen2.5-7B",               "Qwen2.5-7B",   "CLM",  7e9),
]

def safe_id(mid): return mid.replace("/","__")

def first_single_token_id(tokenizer, word):
    for form in (" "+word, word):
        try: ids = tokenizer.encode(form, add_special_tokens=False)
        except Exception: continue
        if len(ids) == 1: return ids[0], form
    return None, None

def filter_targets(tokenizer, words):
    out, forms = {}, {}
    for w in words:
        tid, form = first_single_token_id(tokenizer, w)
        if tid is not None:
            out[w] = tid; forms[w] = form
    return out, forms

def batch_size_for(p):
    if p < 1e9: return 64
    if p < 2.5e9: return 32
    if p < 5e9: return 24
    if p < 8e9: return 16
    return 8


In [ ]:
def build_displaced_text(target, n, frame_tmpl, family, mask_str):
    if n == 0:
        return frame_tmpl.format(mask=mask_str) if family == "MLM" else frame_tmpl
    block = " ".join([target] * n)
    rest  = frame_tmpl.format(mask=mask_str) if family == "MLM" else frame_tmpl
    return f"{block} . {rest}"

def make_input(tokenizer, text, family):
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=True,
                    truncation=True, max_length=512)
    if family == "MLM":
        ids = enc["input_ids"][0]
        mid = tokenizer.mask_token_id
        positions = (ids == mid).nonzero(as_tuple=True)[0]
        if len(positions) == 0: return None, None
        slot = int(positions[0])
    else:
        slot = int(enc["input_ids"].shape[1] - 1)
    return enc, slot

def batch_inputs(tokenizer, texts, family):
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=True,
                    truncation=True, max_length=512)
    slots = []
    if family == "MLM":
        mid = tokenizer.mask_token_id
        for b in range(enc["input_ids"].shape[0]):
            positions = (enc["input_ids"][b] == mid).nonzero(as_tuple=True)[0]
            slots.append(int(positions[0]) if len(positions) else -1)
    else:
        for b in range(enc["input_ids"].shape[0]):
            slots.append(int(enc["attention_mask"][b].sum().item()) - 1)
    return enc, slots

@torch.no_grad()
def score_batch(tokenizer, model, family, texts, target_ids):
    enc, slots = batch_inputs(tokenizer, texts, family)
    enc = {k: v.to("cuda") for k, v in enc.items()}
    out = model(**enc)
    logits = out.logits.float()
    rows = []
    for b in range(logits.shape[0]):
        slot = slots[b]
        if slot < 0:
            rows.append({"target_prob": float("nan"), "target_rank": -1, "entropy": float("nan")})
            continue
        lg = logits[b, slot]
        probs = F.softmax(lg, dim=-1)
        tid = target_ids[b]
        if tid is not None:
            tp = float(probs[tid].item())
            rank = int((probs > probs[tid]).sum().item()) + 1
        else:
            tp = float("nan"); rank = -1
        ent = float(-(probs.clamp_min(1e-12).log() * probs).sum().item())
        rows.append({"target_prob": tp, "target_rank": rank, "entropy": ent})
    return rows

def chunked(seq, k):
    for i in range(0, len(seq), k): yield seq[i:i+k]


In [ ]:
def load_model(mid, family):
    print(f"  loading {mid} ({family}, bfloat16) ...", flush=True)
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=False)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token if tok.eos_token else "[PAD]"
    tok.padding_side = "right"
    Cls = AutoModelForMaskedLM if family == "MLM" else AutoModelForCausalLM
    kwargs = dict(torch_dtype=torch.bfloat16, trust_remote_code=False)
    try:
        m = Cls.from_pretrained(mid, attn_implementation="eager", **kwargs)
    except Exception:
        m = Cls.from_pretrained(mid, **kwargs)
    m = m.to("cuda").eval()
    print(f"  loaded in {time.time()-t0:.1f}s", flush=True)
    return tok, m

def free_model(t, m):
    try: del t, m
    except Exception: pass
    gc.collect(); torch.cuda.empty_cache()


def run_one_lang(model_id, short, family, lang, tokenizer, model, params):
    bs = batch_size_for(params)
    mask_str = tokenizer.mask_token if family == "MLM" else None
    targets, forms = filter_targets(tokenizer, WORDLISTS[lang])
    if len(targets) < 10:
        print(f"  {short}/{lang}: only {len(targets)} single-token-valid; skipping")
        return None
    print(f"  {short}/{lang}: {len(targets)}/{len(WORDLISTS[lang])} valid words")
    frames = FRAMES_MLM[lang] if family == "MLM" else FRAMES_CLM[lang]
    items = []
    for fid, ftmpl in frames:
        for w, tid in targets.items():
            for n in NS:
                items.append({
                    "model": model_id, "family": family, "lang": lang,
                    "frame_id": fid, "target_word": w, "token_id": tid,
                    "N": n, "text": build_displaced_text(w, n, ftmpl, family, mask_str),
                })
    rows = []
    for batch in chunked(items, bs):
        texts = [it["text"] for it in batch]
        tids  = [it["token_id"] for it in batch]
        scores = score_batch(tokenizer, model, family, texts, tids)
        for it, sc in zip(batch, scores):
            r = {**it, **sc}; r.pop("text", None); rows.append(r)
    return pd.DataFrame(rows)


def run_one_model(model_id, short, family, params):
    out_path = ML_DIR / f"displaced/multiling__{safe_id(model_id)}.parquet"
    if out_path.exists():
        print(f"skip {model_id} (already done)")
        return
    print(f"=== {model_id} ({family}, ~{params/1e9:.2f}B) ===")
    t0 = time.time()
    try:
        tok, m = load_model(model_id, family)
        all_rows = []
        for lang in WORDLISTS:
            df = run_one_lang(model_id, short, family, lang, tok, m, params)
            if df is not None:
                all_rows.append(df)
        free_model(tok, m)
        if all_rows:
            full = pd.concat(all_rows, ignore_index=True)
            full.to_parquet(out_path, index=False)
            print(f"  saved {out_path.name}  rows={len(full)}  ({(time.time()-t0)/60:.1f} min)")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()
        try: free_model(tok, m)
        except Exception: pass
        gc.collect(); torch.cuda.empty_cache()

for mid, short, fam, p in ML_MODELS:
    run_one_model(mid, short, fam, p)

print("\n=== multilingual run finished ===")


In [ ]:
import glob

files = sorted(glob.glob(str(ML_DIR / "displaced/multiling__*.parquet")))
print(f"merging {len(files)} files")
disp = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()
if len(disp):
    disp.to_parquet(ML_DIR/"summary"/"multiling_all.parquet", index=False)
    print(f"merged shape={disp.shape}")

# Per-(model, lang, frame) summary
def per_word_drop(disp_df):
    rows = []
    for (m, lg, fr, w), g in disp_df.groupby(["model","lang","frame_id","target_word"]):
        g = g.set_index("N")["target_prob"].sort_index()
        if not all(n in g.index for n in NS_NONZERO): continue
        sub = g.loc[NS_NONZERO]
        if not sub.notna().any(): continue
        peak = float(sub.max(skipna=True))
        if not np.isfinite(peak) or peak <= 0: continue
        if 30 not in g.index or pd.isna(g.loc[30]): continue
        peak_n = int(sub.idxmax(skipna=True))
        p30 = float(g.loc[30])
        rows.append({"model": m, "lang": lg, "frame_id": fr, "target_word": w,
                     "peak_N": peak_n, "peak_P": peak, "P_N30": p30,
                     "drop": (peak - p30)/peak})
    return pd.DataFrame(rows)

if len(disp):
    drop_df = per_word_drop(disp)
    drop_df.to_parquet(ML_DIR/"summary"/"per_word_drops.parquet", index=False)
    print(f"per-word drops: {len(drop_df)} rows")

    # Per-(model, lang, frame) summary with bootstrap CI
    rng = np.random.default_rng(42)
    rows = []
    for (m, lg, fr), g in drop_df.groupby(["model","lang","frame_id"]):
        v = g["drop"].dropna().values
        if len(v) == 0: continue
        boots = [float(rng.choice(v, size=len(v), replace=True).mean()) for _ in range(500)]
        lo, hi = float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
        sub = disp[(disp.model==m) & (disp.lang==lg) & (disp.frame_id==fr)]
        mc = sub.groupby("N")["target_prob"].mean().sort_index()
        nz = mc.loc[[n for n in NS_NONZERO if n in mc.index]].dropna()
        if len(nz) == 0 or 30 not in mc.index or pd.isna(mc.loc[30]): continue
        rows.append({"model": m.split("/")[-1], "lang": lg, "frame_id": fr,
                     "n_words": int(len(v)),
                     "peak_N": int(nz.idxmax()),
                     "peak_P": float(nz.max()),
                     "P_N30": float(mc.loc[30]),
                     "drop_mean": float(v.mean()),
                     "drop_ci_lo": lo, "drop_ci_hi": hi})
    summary = pd.DataFrame(rows).sort_values(["model","lang","frame_id"])
    summary.to_csv(ML_DIR/"summary"/"model_lang_frame_summary.csv", index=False)
    print(summary.to_string(index=False))
